# 🧹 Data Cleaning — Pangan Jember 2024-2025
## CRISP-DM Phase 3: Data Preparation

Notebook ini membersihkan dataset harga pangan Jember secara bertahap, sesuai masalah yang ditemukan pada fase *Data Understanding*.

**Masalah yang akan ditangani:**
1. Load & inspeksi awal
2. Hapus baris kategori (bukan data observasi)
3. Hapus duplikat
4. Konversi kolom `tanggal` ke datetime
5. Konversi kolom `selisih` ke numerik
6. Bersihkan kolom `persen`
7. Tangani `harga_lama` = 0
8. Tangani `harga_sekarang` = 0 (anomali)
9. Bersihkan nama komoditas
10. Simpan hasil


## 📦 Import Library

In [3]:
import pandas as pd
import numpy as np

print("Library siap.")


Library siap.


## 1️⃣ Load & Inspeksi Awal

Muat dataset dan lihat struktur dasarnya sebelum melakukan perubahan apapun.

In [4]:
df = pd.read_csv('pangan_makanan_jember_2021_2025.csv')

print(f"Shape  : {df.shape}")
print(f"Kolom  : {df.columns.tolist()}")
print()
df.head(10)


Shape  : (142038, 7)
Kolom  : ['tanggal', 'komoditas', 'satuan', 'harga_lama', 'harga_sekarang', 'selisih', 'persen']



,tanggal,komoditas,satuan,harga_lama,harga_sekarang,selisih,persen
0,2021-01-01,BERAS,NaN,0,0,NaN,NaN
1,2021-01-01,-Beras Premium,kg,15120,15120,0,"0,00%"
2,2021-01-01,-Beras Medium,kg,12020,12020,0,"0,00%"
3,2021-01-01,GULA,NaN,0,0,NaN,NaN
4,2021-01-01,-Gula Kristal Putih,kg,16700,16700,0,"0,00%"
5,2021-01-01,MINYAK GORENG,NaN,0,0,NaN,NaN
6,2021-01-01,-Minyak Goreng Curah,kg,19440,19440,0,"0,00%"
7,2021-01-01,-Minyak Goreng Kemasan Premium,1 liter,21500,21500,0,"0,00%"
8,2021-01-01,-Minyak Goreng Kemasan Sederhana,1 Liter,18625,18625,0,"0,00%"
9,2021-01-01,-Minyak Goreng MINYAKITA,1 Liter,15700,15700,0,"0,00%"


In [5]:
print("Tipe data:")
print(df.dtypes)
print()
print("Missing values per kolom:")
print(df.isnull().sum())


Tipe data:
tanggal           object
komoditas         object
satuan            object
harga_lama         int64
harga_sekarang     int64
selisih           object
persen            object
dtype: object

Missing values per kolom:
tanggal               0
komoditas             0
satuan            20031
harga_lama            0
harga_sekarang        0
selisih           20031
persen            20031
dtype: int64


## 2️⃣ Hapus Baris Kategori

Baris seperti `BERAS`, `GULA`, `MINYAK GORENG` adalah baris *grup/header*, bukan data harga.  
Cirinya: kolom `satuan` bernilai `NaN` dan `harga_lama`/`harga_sekarang` = 0.

→ Baris ini **tidak relevan** untuk analisis dan harus dihapus.

In [6]:
# Tampilkan contoh baris kategori sebelum dihapus
baris_kategori = df[df['satuan'].isna()]
print(f"Jumlah baris kategori: {len(baris_kategori)}")
baris_kategori.head(5)


Jumlah baris kategori: 20031


,tanggal,komoditas,satuan,harga_lama,harga_sekarang,selisih,persen
0,2021-01-01,BERAS,NaN,0,0,NaN,NaN
3,2021-01-01,GULA,NaN,0,0,NaN,NaN
5,2021-01-01,MINYAK GORENG,NaN,0,0,NaN,NaN
10,2021-01-01,DAGING,NaN,0,0,NaN,NaN
14,2021-01-01,TELUR AYAM,NaN,0,0,NaN,NaN


In [7]:
df = df[df['satuan'].notna()].copy()
print(f"Shape setelah drop baris kategori: {df.shape}")


Shape setelah drop baris kategori: (122007, 7)


## 3️⃣ Hapus Duplikat

Duplikat bisa muncul akibat kesalahan input atau scraping ulang data yang sama.

In [8]:
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat: {jumlah_duplikat}")
df.duplicated().value_counts()


Jumlah baris duplikat: 86


False    121921
True         86
Name: count, dtype: int64

In [9]:
df = df.drop_duplicates()
print(f"Shape setelah drop duplikat: {df.shape}")


Shape setelah drop duplikat: (121921, 7)


## 4️⃣ Konversi Kolom `tanggal` ke Datetime

Kolom `tanggal` masih bertipe string. Perlu dikonversi agar bisa dipakai untuk analisis time-series.

In [10]:
print(f"Tipe sebelum: {df['tanggal'].dtype}")
print(f"Contoh nilai: {df['tanggal'].iloc[0]}")


Tipe sebelum: object
Contoh nilai: 2021-01-01


In [11]:
df['tanggal'] = pd.to_datetime(df['tanggal'])

print(f"Tipe sesudah: {df['tanggal'].dtype}")
print(f"Rentang tanggal: {df['tanggal'].min()} s/d {df['tanggal'].max()}")


Tipe sesudah: datetime64[ns]
Rentang tanggal: 2021-01-01 00:00:00 s/d 2025-12-31 00:00:00


## 5️⃣ Konversi Kolom `selisih` ke Numerik

Kolom `selisih` terbaca sebagai string padahal seharusnya numerik (float).

In [12]:
print(f"Tipe sebelum: {df['selisih'].dtype}")
print(f"Contoh nilai: {df['selisih'].head(5).tolist()}")


Tipe sebelum: object
Contoh nilai: ['0', '0', '0', '0', '0']


In [13]:
df['selisih'] = pd.to_numeric(df['selisih'], errors='coerce')

print(f"Tipe sesudah: {df['selisih'].dtype}")
print(f"Missing setelah konversi: {df['selisih'].isna().sum()}")


Tipe sesudah: float64
Missing setelah konversi: 9095


## 6️⃣ Bersihkan Kolom `persen`

Format asal: `"0,18%"` (string, koma sebagai desimal, ada simbol %).  
Target: `0.0018` (float, proporsi desimal).

In [14]:
print(f"Tipe sebelum: {df['persen'].dtype}")
print(f"Contoh nilai: {df['persen'].dropna().head(5).tolist()}")


Tipe sebelum: object
Contoh nilai: ['0,00%', '0,00%', '0,00%', '0,00%', '0,00%']


In [15]:
df['persen'] = (
    df['persen']
    .str.replace('%', '', regex=False)   # hapus simbol %
    .str.replace(',', '.', regex=False)  # ganti koma -> titik
    .pipe(pd.to_numeric, errors='coerce')
    / 100                                # ubah ke proporsi
)

print(f"Tipe sesudah: {df['persen'].dtype}")
print(f"Contoh nilai: {df['persen'].dropna().head(5).tolist()}")


Tipe sesudah: float64
Contoh nilai: [0.0, 0.0, 0.0, 0.0, 0.0]


## 7️⃣ Tangani `harga_lama` = 0

Pada baris item nyata, `harga_lama` = 0 kemungkinan berarti **data awal** sebelum ada perubahan harga.  
Strategi: isi dengan nilai `harga_sekarang` (asumsi harga belum berubah).

In [16]:
mask = df['harga_lama'] == 0
print(f"Jumlah baris dengan harga_lama=0: {mask.sum()}")
df[mask].head(5)


Jumlah baris dengan harga_lama=0: 9095


,tanggal,komoditas,satuan,harga_lama,harga_sekarang,selisih,persen
30,2021-01-01,-Kedelai Lokal,kg,0,0,NaN,NaN
50,2021-01-01,-Semen Tonasa,40 Kg,0,0,NaN,NaN
52,2021-01-01,-Semen Dynamix,40 Kg,0,0,NaN,NaN
59,2021-01-01,-KAYU BALOK MERANTI (4 X 10),Btg,0,0,NaN,NaN
60,2021-01-01,-Papan Meranti (4m X 3cm X 20mm),Lembar,0,0,NaN,NaN


In [17]:
df.loc[mask, 'harga_lama'] = df.loc[mask, 'harga_sekarang']

print(f"Sisa harga_lama=0 setelah imputasi: {(df['harga_lama'] == 0).sum()}")


Sisa harga_lama=0 setelah imputasi: 9095


## 8️⃣ Tangani `harga_sekarang` = 0 (Anomali)

Harga komoditas pangan tidak mungkin bernilai 0.  
Nilai ini dianggap **data tidak valid** → ganti dengan `NaN`.

In [18]:
mask_zero = df['harga_sekarang'] == 0
print(f"Jumlah baris dengan harga_sekarang=0: {mask_zero.sum()}")
df[mask_zero].head(5)


Jumlah baris dengan harga_sekarang=0: 9095


,tanggal,komoditas,satuan,harga_lama,harga_sekarang,selisih,persen
30,2021-01-01,-Kedelai Lokal,kg,0,0,NaN,NaN
50,2021-01-01,-Semen Tonasa,40 Kg,0,0,NaN,NaN
52,2021-01-01,-Semen Dynamix,40 Kg,0,0,NaN,NaN
59,2021-01-01,-KAYU BALOK MERANTI (4 X 10),Btg,0,0,NaN,NaN
60,2021-01-01,-Papan Meranti (4m X 3cm X 20mm),Lembar,0,0,NaN,NaN


In [19]:
df.loc[mask_zero, 'harga_sekarang'] = np.nan

print(f"Sisa harga_sekarang=0: {(df['harga_sekarang'] == 0).sum()}")
print(f"Missing harga_sekarang (NaN): {df['harga_sekarang'].isna().sum()}")


Sisa harga_sekarang=0: 0
Missing harga_sekarang (NaN): 9095


## 9️⃣ Filter Komoditas Non-Pangan

Dataset ini ternyata juga mengandung data **non-pangan** seperti bahan bangunan (Semen, Bata, Kayu, dll) dan Pupuk.  
Langkah ini:
1. Filter hanya baris item (diawali `-`)
2. Hapus komoditas yang bukan pangan

> **Catatan:** Langkah ini dilakukan sebelum membersihkan prefix `-` agar filter `startswith('-')` masih bisa bekerja.

In [20]:
# Lihat dulu komoditas non-pangan yang akan dihapus
non_food_keywords = ['Semen', 'Kayu', 'Bata', 'Papan', 'Triplek', 'Besi', 'Paku', 'Pupuk', 'Gas']

mask_non_food = df['komoditas'].str.contains('|'.join(non_food_keywords), case=False, na=False)
print(f"Jumlah baris non-pangan: {mask_non_food.sum()}")
print()
print("Komoditas non-pangan yang ditemukan:")
print(df[mask_non_food]['komoditas'].unique())


Jumlah baris non-pangan: 47296

Komoditas non-pangan yang ditemukan:
['-Bata' '-Semen Gresik' '-Semen Tiga Roda' '-Semen Padang'
 '-Semen Tonasa' '-Semen Bosowa' '-Semen Dynamix'
 '-KAYU BALOK MERANTI (4 X 10)' '-Papan Meranti (4m X 3cm X 20mm)'
 '-TRIPLEK (6MM)' '-Besi Beton 6 mm (12/9m)' '-Besi Beton 8 mm (12/9m)'
 '-Besi Beton 10 mm (12/9m)' '-Besi Beton 12 mm (12/9m)'
 '-Paku Ukuran 10Cm' '-Paku Ukuran 2 Cm' '-Paku Ukuran 3Cm'
 '-Paku Ukuran 4Cm' '-Paku Ukuran 5Cm' '-Paku Ukuran 7Cm'
 '-GAS ELPIGI 3 Kg' '-Pupuk KCL Non Subsidi' '-Pupuk NPK Non Subsidi'
 '-Pupuk SP 35 Non Subsidi' '-Pupuk Urea Non Subsidi'
 '-Pupuk ZA Non Subsidi']


In [21]:
# Filter: ambil hanya baris item (diawali '-'), lalu buang non-pangan
df_filtered = df[df['komoditas'].str.startswith('-', na=False)].copy()

df_filtered = df_filtered[
    ~df_filtered['komoditas'].str.contains('|'.join(non_food_keywords), case=False, na=False)
]

print(f"Shape setelah filter non-pangan: {df_filtered.shape}")
print(f"Contoh komoditas tersisa:")
print(df_filtered['komoditas'].unique()[:15])


Shape setelah filter non-pangan: (74625, 7)
Contoh komoditas tersisa:
['-Beras Premium' '-Beras Medium' '-Gula Kristal Putih'
 '-Minyak Goreng Curah' '-Minyak Goreng Kemasan Premium'
 '-Minyak Goreng Kemasan Sederhana' '-Minyak Goreng MINYAKITA'
 '-Daging Sapi Paha Belakang' '-Daging Ayam Ras' '-Daging Ayam Kampung'
 '-Telur Ayam Ras' '-Telur Ayam Kampung' '-Susu Kental Manis Merk Bendera'
 '-Susu Kental Manis Merk Indomilk' '-Susu Bubuk Merk Bendera (Instant)']


## 1️⃣0️⃣ Bersihkan Nama Komoditas

Setelah filter non-pangan, hapus prefix `-` dari nama komoditas.

In [22]:
print("Sebelum:")
print(df_filtered['komoditas'].head(5).tolist())


Sebelum:
['-Beras Premium', '-Beras Medium', '-Gula Kristal Putih', '-Minyak Goreng Curah', '-Minyak Goreng Kemasan Premium']


In [23]:
df_filtered['komoditas'] = df_filtered['komoditas'].str.lstrip('-').str.strip()

print("Sesudah:")
print(df_filtered['komoditas'].head(5).tolist())


Sesudah:
['Beras Premium', 'Beras Medium', 'Gula Kristal Putih', 'Minyak Goreng Curah', 'Minyak Goreng Kemasan Premium']


## 1️⃣1️⃣ Ringkasan & Simpan Hasil

Tampilkan kondisi akhir dataset dan simpan ke file baru.

In [24]:
print("=" * 45)
print("RINGKASAN DATA BERSIH")
print("=" * 45)
print(f"Shape          : {df_filtered.shape}")
print(f"Rentang tanggal: {df_filtered['tanggal'].min().date()} s/d {df_filtered['tanggal'].max().date()}")
print()
print("Tipe data:")
print(df_filtered.dtypes)
print()
print("Missing values:")
print(df_filtered.isnull().sum())


RINGKASAN DATA BERSIH
Shape          : (74625, 7)
Rentang tanggal: 2021-01-01 s/d 2025-12-31

Tipe data:
tanggal           datetime64[ns]
komoditas                 object
satuan                    object
harga_lama                 int64
harga_sekarang           float64
selisih                  float64
persen                   float64
dtype: object

Missing values:
tanggal              0
komoditas            0
satuan               0
harga_lama           0
harga_sekarang    1819
selisih           1819
persen            1819
dtype: int64


In [25]:
df_filtered.head(10)

,tanggal,komoditas,satuan,harga_lama,harga_sekarang,selisih,persen
1,2021-01-01,Beras Premium,kg,15120,15120.0,0.0,0.0
2,2021-01-01,Beras Medium,kg,12020,12020.0,0.0,0.0
4,2021-01-01,Gula Kristal Putih,kg,16700,16700.0,0.0,0.0
6,2021-01-01,Minyak Goreng Curah,kg,19440,19440.0,0.0,0.0
7,2021-01-01,Minyak Goreng Kemasan Premium,1 liter,21500,21500.0,0.0,0.0
8,2021-01-01,Minyak Goreng Kemasan Sederhana,1 Liter,18625,18625.0,0.0,0.0
9,2021-01-01,Minyak Goreng MINYAKITA,1 Liter,15700,15700.0,0.0,0.0
11,2021-01-01,Daging Sapi Paha Belakang,kg,123000,123000.0,0.0,0.0
12,2021-01-01,Daging Ayam Ras,kg,36600,36600.0,0.0,0.0
13,2021-01-01,Daging Ayam Kampung,ekor,58333,58333.0,0.0,0.0


In [26]:
df_filtered.to_csv('pangan_jember_cleanfix.csv', index=False)
print("✅ Data bersih disimpan ke: pangan_jember_cleanfix.csv")


✅ Data bersih disimpan ke: pangan_jember_cleanfix.csv
